<a href="https://colab.research.google.com/github/Steve-Golinski/choose_llm/blob/main/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Directly executing some smaller models for inference on the GPU
## ..using the Hugging Face Hub and Transformers library


Finally, this is the other way to show you direct inference of a model. We download the same 3 models from Hugging Face. We then have similar code to run each one.

You can run this for the famous Llama 3 model from Meta (the 8G version) but it will take a little longer. Before you can access it, you need to visit their model page on Hugging face at https://huggingface.co/meta-llama/Meta-Llama-3-8B and accept their terms of use. You should get an email confirming your access about 30 mins later.

In [1]:
# pip installs

!pip install -q requests==2.31.0 torch bitsandbytes transformers sentencepiece accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.1/314.1 kB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 71.6 MB/s eta 0:00:00


In [2]:
# imports

import os
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer, BitsAndBytesConfig

In [11]:
# Log in to HuggingFace
# Either add your token into the secrets of this notebook using the Key icon,
# Or replace the next line


hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [12]:
# And also the same models as last time

PHI3_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
QWEN2_MODEL_NAME = "Qwen/Qwen2-7B-Instruct"
STARCODER2_MODEL_NAME = "bigcode/starcoder2-3b"

# Also for the exercise after the live training

LLAMA3_MODEL_NAME = "meta-llama/Meta-Llama-3-8B"

## Hugging Face's high level API: the Pipeline

In [13]:
# Set up the pipeline - a Pipeline abstracts a series of steps with a very simple API

pipeline = transformers.pipeline("text-generation", model=PHI3_MODEL_NAME, device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [14]:
# Use the pipeline

prompt = """<|user|>
Tell a light joke for a room full of data scientists<|end|>
<|assistant|>"""

pipeline(prompt, max_new_tokens=20)

You are not running the flash-attention implementation, expect numerical differences.


[{'generated_text': "<|user|>\nTell a light joke for a room full of data scientists<|end|>\n<|assistant|> Why don't data scientists ever play hide and seek?\n\nBecause good luck hiding"}]

# The lower level Transformers API

## The Tokenizer and The Model

For each of our models, we will be creating 2 important objects from the Hugging Face transformers library:

### A Tokenizer

This maps between text and tokens. Different models might have differences between their use of tokens, such as different vocab or special characters.

Methods like `encode` and `decode` can be used to convert between text and tokens.

For more, see:

https://huggingface.co/docs/transformers/en/main_classes/tokenizer

### A Model

This is the key object which is a subclass of PreTrainedModel that can be loaded from the hub, used for inference and training, and saved to the hub.

https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/model#transformers.PreTrainedModel

In [15]:
# Create the Tokenizer and Model for Microsoft Phi 3

phi3_tokenizer = AutoTokenizer.from_pretrained(PHI3_MODEL_NAME, trust_remote_code=True)
phi3_tokenizer.pad_token = phi3_tokenizer.eos_token
phi3_tokenizer.padding_side = "right"
quant_config = BitsAndBytesConfig(load_in_8bit=True)

phi3 = AutoModelForCausalLM.from_pretrained(PHI3_MODEL_NAME, quantization_config=quant_config, device_map="auto")
phi3.generation_config.pad_token_id = phi3_tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
# To show the tokenizer at work

text = 'Tell a light joke for a room full of data scientists'
tokens = phi3_tokenizer.encode(text)
print(f'There are {len(text)} characters in the text')
print(f'There are {len(tokens)} tokens in the tokenized version')
print('Here are the tokens:')
print(tokens)

There are 52 characters in the text
There are 13 tokens in the tokenized version
Here are the tokens:
[24948, 263, 3578, 2958, 446, 363, 263, 5716, 2989, 310, 848, 9638, 2879]


In [17]:
# And to reconstitute the original text

recreated_text = phi3_tokenizer.decode(tokens)
print(recreated_text)

Tell a light joke for a room full of data scientists


In [18]:
# Or to see how each token is mapped

recreated_text_by_token = phi3_tokenizer.batch_decode(tokens)
print(recreated_text_by_token)

['Tell', 'a', 'light', 'jo', 'ke', 'for', 'a', 'room', 'full', 'of', 'data', 'scient', 'ists']


In [19]:
# Taking a look at the vocab

vocab = phi3_tokenizer.vocab

print(f"There are {len(vocab):,} tokens in Phi3's vocab and it looks like this:\n")
print(vocab)

There are 32,011 tokens in Phi3's vocab and it looks like this:

{'▁ER': 8982, '▁poem': 26576, '▁соста': 15553, 'vity': 17037, ',”': 3995, '▁instinct': 26508, '▁ори': 19974, 'лання': 12867, '▁escaped': 19824, '▁just': 925, 'story': 24098, '▁proget': 6342, '▁costru': 18979, '▁Vater': 19828, '▁agricult': 18032, 'Only': 11730, 'rec': 3757, '▁aussi': 5695, '▁Emer': 27718, 'iks': 29292, 'building': 25237, 'quire': 1548, '▁place': 2058, '▁constitution': 16772, '▁Self': 21782, '▁Div': 4910, '▁thin': 16835, 'ieval': 16837, '▁elli': 22434, 'ès': 2093, 'ality': 2877, '▁Tre': 6479, 'History': 20570, '▁стар': 14871, '▁Kraft': 25818, 'теля': 11146, '▁minimal': 13114, 'ída': 28815, '▁Ky': 16931, "'$": 13090, '▁Constantin': 23194, '▁carefully': 16112, 'Change': 7277, '▁destac': 18583, 'ità': 3943, '▁magazine': 14853, 'шение': 18907, 'бро': 20176, '▁музы': 16925, 'firebase': 17445, '...)': 11410, 'oure': 25664, 'plus': 11242, 'AndroidRuntime': 12746, '▁dedicated': 16955, 'чної': 20705, '▁sugar': 26438

In [20]:
# And let's look at the model

phi3

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear8bitLt(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear8bitLt(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear8bitLt(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features

In [21]:
# Now create the Tokenizer and Model for Qwen2 from Alibaba Cloud - 7B params

qwen2_tokenizer = AutoTokenizer.from_pretrained(QWEN2_MODEL_NAME, trust_remote_code=True)
qwen2_tokenizer.pad_token = qwen2_tokenizer.eos_token
qwen2_tokenizer.padding_side = "right"
quant_config = BitsAndBytesConfig(load_in_8bit=True)

qwen2 = AutoModelForCausalLM.from_pretrained(QWEN2_MODEL_NAME, quantization_config=quant_config, device_map="auto")
qwen2.generation_config.pad_token_id = qwen2_tokenizer.pad_token_id

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [22]:
# # Create the Tokenizer and Model for Starcoder 2 from Big Code

starcoder2_tokenizer = AutoTokenizer.from_pretrained(STARCODER2_MODEL_NAME, trust_remote_code=True)
starcoder2_tokenizer.pad_token = starcoder2_tokenizer.eos_token
starcoder2_tokenizer.padding_side = "right"
quant_config = BitsAndBytesConfig(load_in_8bit=True)

starcoder2 = AutoModelForCausalLM.from_pretrained(STARCODER2_MODEL_NAME, quantization_config=quant_config, device_map="auto")
starcoder2.generation_config.pad_token_id = starcoder2_tokenizer.pad_token_id

tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

In [23]:
# Craft the prompt in the right format for Phi 3
# You'll notice the use of special tokens; these were used during training,
# So for best results we need to replicate the same structure now

messages = [{"role": "user", "content": "Tell a light joke for a room full of data scientists"}]
prompt = phi3_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

<|user|>
Tell a light joke for a room full of data scientists<|end|>
<|assistant|>



In [24]:
# Ask Phi 3 for a joke

streamer = TextStreamer(phi3_tokenizer)
inputs = phi3_tokenizer.encode(prompt, return_tensors="pt").to("cuda")
outputs = phi3.generate(inputs, max_new_tokens=200, streamer=streamer)

<|user|> Tell a light joke for a room full of data scientists<|end|><|assistant|> Why did the data scientist break up with the algorithm?


Because it was always calculating the odds and never taking any risks!<|end|>


In [25]:
# Craft the prompt in the right format for Qwen2
# Notice that the special token usage is different

messages = [{"role": "user", "content": "Tell a light joke for a room full of data scientists"}]
prompt = qwen2_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Tell a light joke for a room full of data scientists<|im_end|>
<|im_start|>assistant



In [26]:
# Ask Qwen 2 for a joke

streamer = TextStreamer(qwen2_tokenizer)
inputs = qwen2_tokenizer.encode(prompt, return_tensors="pt").to("cuda")
outputs = qwen2.generate(inputs, max_new_tokens=200, streamer=streamer)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Tell a light joke for a room full of data scientists<|im_end|>
<|im_start|>assistant
Why do data scientists prefer dark chocolate over milk chocolate?

Because they like their chocolate with high cocoa content!<|im_end|>


In [27]:
# Enough with the jokes! Let's ask Starcoder2 to write a hello_world method

prompt = "def hello_world():\n"
streamer = TextStreamer(starcoder2_tokenizer)
inputs = starcoder2_tokenizer.encode(prompt, return_tensors="pt").to("cuda")
outputs = starcoder2.generate(inputs, max_new_tokens=200, streamer=streamer)

def hello_world():
	return "Hello World"

@app.route('/hello/<name>')
def hello_name(name):
	return "Hello %s" % name

@app.route('/hello/<name>/<int:age>')
def hello_name_age(name, age):
	return "Hello %s, you are %d years old" % (name, age)

@app.route('/hello/<name>/<int:age>/<int:height>')
def hello_name_age_height(name, age, height):
	return "Hello %s, you are %d years old, and your height is %d" % (name, age, height)

@app.route('/hello/<name>/<int:age>/<int:height>/<int:weight>')
def hello_name_age_height_weight(name, age, height, weight):
	return "Hello %s, you


## Exercises:

1. Now go and repeat this for the famous Llama 3 8G model.

2. Also, for another example of the Hugging Face High Level Pipeline API but for something completely different - text to image generation using a diffusion model - check out this:

https://colab.research.google.com/drive/1N1kDINhcdrjabkSCW9UpblkIMWyfZBsR?usp=sharing